
# CyberHackAI — phishing URL classification, V8

Эта версия пересобрана так, чтобы **запускаться с нуля только из `train.csv`, `test.csv`, `sample_submission.csv`**
Главные изменения относительно V7:

1. **5-fold Stratified OOF** для базовых текстовых моделей.
2. Усреднение `test decision score` по пяти fold-моделям.
3. **Character hashing + fold-local TF-IDF на 262 144 char-признаках** (`2**18`) + 65 536 word-признаков.
4. Отдельные модели для `host/domain` и `path/query`.
5. **SimHash-like LSH similarity features**: близкие URL ловятся по 64-битной n-gram сигнатуре и train-only band statistics, без exact-match leakage.
6. Fold-safe reputation/target-encoding признаки для `host`, `root`, `first_path`, `path_template`, `host × first_path`, `root × path_template`.
7. Более глубокий **CatBoost** с категориальными комбинациями и early stopping.
8. OOF meta-stack и **единый threshold, подобранный по pooled OOF всех folds**.
9. Вторая, отличающаяся по ошибкам модель `text + CatBoost` для выбора двух решений на private.
10. Exact train/test URL lookup применяется **только после** модельного предсказания и никогда не участвует в CV/threshold tuning.

> Цель — максимально приблизиться к 0.98 F1 честной OOF-метрикой

Ожидаемое окружение пользователя: **Python 3.14.0, pandas 2.3.3**.


In [3]:

# 0. Импорты и конфигурация
import os, re, gc, sys, json, math, pickle, platform, time
from urllib.parse import urlsplit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit, logit
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, precision_recall_curve, confusion_matrix
)
from sklearn.cluster import MiniBatchKMeans
from sklearn.utils.validation import check_is_fitted

try:
    from catboost import CatBoostClassifier
except ImportError as e:
    raise ImportError(
        "Нужен catboost. Установите зависимости из requirements_v8_py314.txt"
    ) from e

SEED = 42
N_SPLITS = 5

# Character hashing 2**18 = 262 144 признака — внутри требуемого диапазона 250–300k.
CHAR_HASH_FEATURES = 2**18
WORD_HASH_FEATURES = 2**16
HOST_HASH_FEATURES = 2**16
PATH_HASH_FEATURES = 2**17

# Полный режим по умолчанию.
# Для быстрой проверки кода можно временно поставить FAST_MODE=True.
FAST_MODE = False

CATBOOST_ITERATIONS = 1800 if not FAST_MODE else 80
CATBOOST_DEPTH = 9 if not FAST_MODE else 7
CATBOOST_OD_WAIT = 100 if not FAST_MODE else 20

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("platform:", platform.platform())
print("CPU:", os.cpu_count())
print("FAST_MODE:", FAST_MODE)

# Код не использует устаревшие DataFrame.append / Series.append и совместим с pandas 2.3.x.


Python: 3.14.0
pandas: 2.3.3
platform: Windows-11-10.0.26200-SP0
CPU: 10
FAST_MODE: False


In [2]:
pip install catboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 462.1 kB/s eta 0:03:40
   ---------------------------------------- 0.5/101.7 MB 462.1 kB/s eta 0:03:40
   ---------------------------------------- 0.5/101.7 MB 462.1 kB/s eta 0:03:40
   ---------------------------------------- 0.8/101.7 MB 419.3 kB/s eta 0:04:01
   ---------------------------------------- 0.8/101.7 MB 419.3 kB/s eta 0:04:01
   ---------------------------------------- 0.8/101.7 MB 419.3 kB/s eta 0:04:01
   --------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

# 1. Загрузка данных
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_PATH = "sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

required_train = {"Id", "url", "Predicted"}
required_test = {"Id", "url"}
assert required_train.issubset(train.columns)
assert required_test.issubset(test.columns)
assert {"Id", "Predicted"}.issubset(sample_submission.columns)

train["url"] = train["url"].fillna("").astype(str)
test["url"] = test["url"].fillna("").astype(str)
train["Predicted"] = train["Predicted"].astype(int)

print("train:", train.shape)
print("test :", test.shape)
display(train.head())


train: (185910, 3)
test : (46478, 2)


,Id,url,Predicted
0,0,http://banqsuepoy.temp.swtest.ru/pb/assistance...,1
1,1,https://my.mail.ru/community/mir24.tv/,0
2,2,https://rmailidtrack-b484fa.ingress-bonde.ewp....,1
3,3,http://tjvodxie.cn.b2b168.com/m296765p1/,0
4,4,https://articulate.com/360/review,0


In [5]:

# 2. Очистка: конфликты меток и точные дубли
label_nunique = train.groupby("url", sort=False)["Predicted"].nunique()
conflict_urls = set(label_nunique[label_nunique > 1].index)

clean = (
    train.loc[~train["url"].isin(conflict_urls)]
         .drop_duplicates("url", keep="first")
         .reset_index(drop=True)
)

print("Исходных строк:", len(train))
print("URL с конфликтующими метками:", len(conflict_urls))
print("Уникальных непротиворечивых URL:", len(clean))
print("Доля phishing:", clean["Predicted"].mean())

# Exact lookup строим отдельно и НЕ используем в OOF.
exact_lookup = (
    train.groupby("url", sort=False)["Predicted"]
         .agg(lambda s: int(s.iloc[0]) if s.nunique() == 1 else np.nan)
         .dropna()
         .astype(int)
         .to_dict()
)


Исходных строк: 185910
URL с конфликтующими метками: 57
Уникальных непротиворечивых URL: 175522
Доля phishing: 0.4774216337553127


In [6]:

# 3. Разбор URL и ручные признаки

SUSPICIOUS_WORDS = [
    "login","verify","secure","account","update","bank","confirm","signin",
    "bonus","gift","password","wallet","invoice","support","auth","recover",
    "security","credential","webscr","payment","billing","unlock","validate"
]
BRANDS = [
    "paypal","apple","microsoft","google","facebook","instagram","amazon",
    "netflix","visa","mastercard","outlook","office","icloud","wellsfargo"
]

def parse_url_parts(url: str):
    raw = str(url).strip()
    candidate = raw if "://" in raw else "http://" + raw
    try:
        p = urlsplit(candidate)
        host = (p.hostname or "").lower().strip(".")
        path = (p.path or "").lower()
        query = (p.query or "").lower()
        port = p.port
    except Exception:
        host, path, query, port = "", "", "", None

    labels = [x for x in host.split(".") if x]
    root = ".".join(labels[-2:]) if len(labels) >= 2 else host
    tld = labels[-1] if len(labels) >= 2 else ""
    first_path = next((x for x in path.split("/") if x), "")[:64]

    # Шаблон пути: ID/hex/digit sequences заменяются, чтобы ловить phishing templates.
    path_template = re.sub(r"[0-9a-f]{8,}", "{id}", path)
    path_template = re.sub(r"\d+", "{n}", path_template)
    path_template = re.sub(r"[^a-z0-9{}._/\-]+", "", path_template)[:180]

    return host, root, tld, first_path, path, query, path_template, port

def build_parts(urls):
    rows = [parse_url_parts(u) for u in urls]
    df = pd.DataFrame(rows, columns=[
        "host","root","tld","first_path","path","query","path_template","port"
    ])
    df["path_query"] = (df["path"] + np.where(df["query"].eq(""), "", "?" + df["query"])).str.slice(0, 700)
    df["host_first"] = (df["host"] + "|" + df["first_path"]).str.slice(0, 220)
    df["root_path_template"] = (df["root"] + "|" + df["path_template"]).str.slice(0, 280)
    return df

def shannon_entropy_text(s: str) -> float:
    if not s:
        return 0.0
    vals = np.fromiter((s.count(ch) for ch in set(s)), dtype=np.float64)
    p = vals / vals.sum()
    return float(-(p * np.log2(p)).sum())

def longest_regex_run(s, pattern):
    found = re.findall(pattern, s)
    return max((len(x) for x in found), default=0)

def build_manual_features(urls, parts):
    urls = pd.Series(urls, dtype="string").fillna("")
    low = urls.str.lower()

    out = pd.DataFrame(index=np.arange(len(urls)))
    out["url_len"] = urls.str.len().to_numpy()
    out["host_len"] = parts["host"].str.len().to_numpy()
    out["path_len"] = parts["path"].str.len().to_numpy()
    out["query_len"] = parts["query"].str.len().to_numpy()

    for name, pat in [
        ("dots", r"\."), ("hyphens", r"-"), ("underscores", r"_"),
        ("slashes", r"/"), ("question_marks", r"\?"), ("equals", r"="),
        ("ampersands", r"&"), ("ats", r"@"), ("percents", r"%")
    ]:
        out[name] = urls.str.count(pat).to_numpy()

    out["digits"] = urls.str.count(r"\d").to_numpy()
    out["letters"] = urls.str.count(r"[A-Za-zА-Яа-яЁё]").to_numpy()
    out["host_depth"] = parts["host"].str.count(r"\.").to_numpy() + 1
    out["https"] = low.str.startswith("https://").astype("int8").to_numpy()
    out["punycode"] = low.str.contains("xn--", regex=False).astype("int8").to_numpy()
    out["is_ip"] = parts["host"].str.match(r"^(?:\d{1,3}\.){3}\d{1,3}$").astype("int8").to_numpy()
    out["has_port"] = parts["port"].notna().astype("int8").to_numpy()
    out["percent_encoding"] = low.str.count(r"%[0-9a-f]{2}").to_numpy()
    out["double_slash_after_scheme"] = (
        low.str.replace(r"^[a-z]+://", "", regex=True).str.contains("//", regex=False)
    ).astype("int8").to_numpy()
    out["http_in_path"] = parts["path"].str.contains("http", regex=False).astype("int8").to_numpy()

    out["suspicious_hits"] = np.zeros(len(out), dtype=np.int16)
    out["brand_hits"] = np.zeros(len(out), dtype=np.int16)
    out["brand_in_path"] = np.zeros(len(out), dtype=np.int16)
    for w in SUSPICIOUS_WORDS:
        out["suspicious_hits"] += low.str.contains(w, regex=False).astype("int16").to_numpy()
    for b in BRANDS:
        out["brand_hits"] += low.str.contains(b, regex=False).astype("int16").to_numpy()
        out["brand_in_path"] += (
            parts["path"].str.contains(b, regex=False) |
            parts["query"].str.contains(b, regex=False)
        ).astype("int16").to_numpy()

    # Эти три признака считаются обычным Python-циклом, но всего один раз.
    out["longest_digit_run"] = [longest_regex_run(x, r"\d+") for x in urls]
    out["longest_token"] = [longest_regex_run(x, r"[A-Za-z0-9]+") for x in urls]
    out["entropy"] = [shannon_entropy_text(x) for x in urls]

    return out.astype(np.float32)

parts = build_parts(clean["url"].to_numpy())
parts_test = build_parts(test["url"].to_numpy())
manual = build_manual_features(clean["url"].to_numpy(), parts)
manual_test = build_manual_features(test["url"].to_numpy(), parts_test)

print("manual features:", manual.shape)
display(manual.describe().T[["mean","std","min","max"]].head(12))


manual features: (175522, 29)


,mean,std,min,max
url_len,55.739521,46.184933,4.0,2048.0
host_len,18.998119,10.722979,0.0,179.0
path_len,24.223118,31.231855,0.0,1984.0
query_len,5.048729,31.031080,0.0,1100.0
dots,2.136735,1.415752,0.0,38.0
hyphens,1.452393,2.721252,0.0,46.0
underscores,0.159057,0.866580,0.0,70.0
slashes,4.117672,2.388963,0.0,84.0
question_marks,0.072487,0.270919,0.0,5.0
equals,0.150676,0.800714,0.0,91.0


In [ ]:

# 4. EDA, корреляции и простая кластеризация
eda = pd.DataFrame({
    "rows": [len(clean)],
    "phishing_share": [clean["Predicted"].mean()],
    "unique_hosts": [parts["host"].nunique()],
    "unique_roots": [parts["root"].nunique()],
    "test_rows": [len(test)],
})
display(eda)

corr = manual.copy()
corr["target"] = clean["Predicted"].to_numpy()
target_corr = corr.corr(numeric_only=True)["target"].drop("target").sort_values(key=np.abs, ascending=False)
display(target_corr.head(20).to_frame("corr_with_target"))

# KMeans здесь нужен как исследовательская часть, а не как финальный классификатор.
from sklearn.preprocessing import StandardScaler
cluster_sample = manual.sample(min(30000, len(manual)), random_state=SEED)
cluster_scaled = StandardScaler().fit_transform(cluster_sample)
kmeans = MiniBatchKMeans(n_clusters=4, random_state=SEED, batch_size=2048, n_init="auto")
cluster_ids = kmeans.fit_predict(cluster_scaled)
print("Размеры кластеров:", np.bincount(cluster_ids))


,rows,phishing_share,unique_hosts,unique_roots,test_rows
0,175522,0.477422,86502,51059,46478


,corr_with_target
host_len,0.476156
https,-0.461673
suspicious_hits,0.268706
question_marks,0.221447
path_len,-0.214558
host_depth,0.198217
hyphens,-0.196129
longest_token,0.184884
dots,0.170712
slashes,-0.147294


Размеры кластеров: [16787  4042  5354  3817]



## 5. SimHash-like LSH similarity без exact-match leakage

Используем 64 signed hashing dimensions по char n-grams. Знак каждой координаты образует 64-битный fingerprint. Он делится на четыре 16-битных band-а.

Для каждого validation/test URL на каждом fold считаются **только статистики band-ов из train-fold**: сглаженная phishing rate и число соседей в том же bucket. Это приблизительный similarity/nearest-neighbor сигнал, но он не видит метку самой validation-строки и не использует exact URL lookup.


In [8]:

# 5. 64-bit SimHash-like fingerprints -> 4 bands x 16 bit
simhash_vec = HashingVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    n_features=64,
    alternate_sign=True,
    norm=None,
    lowercase=True,
    dtype=np.float32,
)

signature_text = (parts["root"] + "/" + parts["path_template"]).fillna("")
signature_test = (parts_test["root"] + "/" + parts_test["path_template"]).fillna("")

def make_lsh_bands(text_series):
    dense = simhash_vec.transform(text_series).toarray()
    bits = (dense > 0).astype(np.uint8)
    packed = np.packbits(bits, axis=1)  # 8 bytes = 64 bits
    bands = np.empty((len(text_series), 4), dtype=np.uint16)
    for j in range(4):
        bands[:, j] = (packed[:, 2*j].astype(np.uint16) << 8) | packed[:, 2*j + 1].astype(np.uint16)
    return bands

lsh_bands = make_lsh_bands(signature_text)
lsh_bands_test = make_lsh_bands(signature_test)

print(lsh_bands.shape, lsh_bands.dtype)


(175522, 4) uint16


In [9]:

# 6. Вспомогательные функции для fold-safe target/reputation и pooled threshold

def best_f1_threshold(y_true, scores):
    p, r, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * p * r / (p + r + 1e-15)
    if len(thresholds) == 0:
        return 0.5, float(f1[0])
    i = int(np.nanargmax(f1[:-1]))
    return float(thresholds[i]), float(f1[i])

def smoothed_mapping(train_key, train_y, alpha=8.0):
    tmp = pd.DataFrame({"key": pd.Series(train_key, dtype="string"), "y": train_y})
    stats = tmp.groupby("key", sort=False)["y"].agg(["sum", "count"])
    global_rate = float(np.mean(train_y))
    rate = (stats["sum"] + alpha * global_rate) / (stats["count"] + alpha)
    return rate, stats["count"], global_rate

def apply_mapping(values, rate_map, count_map, global_rate):
    s = pd.Series(values, dtype="string")
    rate = s.map(rate_map).fillna(global_rate).to_numpy(np.float32)
    count = np.log1p(s.map(count_map).fillna(0).to_numpy(np.float32))
    return rate, count

def fold_safe_reputation(train_idx, val_idx, y, parts, parts_test):
    cols = ["host","root","first_path","path_template","host_first","root_path_template"]
    val_feats, test_feats, names = [], [], []
    for col in cols:
        rate_map, count_map, global_rate = smoothed_mapping(parts[col].iloc[train_idx], y[train_idx], alpha=8.0)
        vr, vc = apply_mapping(parts[col].iloc[val_idx], rate_map, count_map, global_rate)
        tr, tc = apply_mapping(parts_test[col], rate_map, count_map, global_rate)
        val_feats.extend([vr, vc]); test_feats.extend([tr, tc])
        names.extend([f"{col}_rate", f"{col}_logcount"])
    return np.column_stack(val_feats), np.column_stack(test_feats), names

def fold_safe_lsh(train_idx, val_idx, y, bands, bands_test):
    val_feats, test_feats, names = [], [], []
    for b in range(bands.shape[1]):
        train_key = bands[train_idx, b].astype(str)
        val_key = bands[val_idx, b].astype(str)
        test_key = bands_test[:, b].astype(str)
        rate_map, count_map, global_rate = smoothed_mapping(train_key, y[train_idx], alpha=12.0)
        vr, vc = apply_mapping(val_key, rate_map, count_map, global_rate)
        tr, tc = apply_mapping(test_key, rate_map, count_map, global_rate)
        val_feats.extend([vr, vc]); test_feats.extend([tr, tc])
        names.extend([f"lsh_band{b}_rate", f"lsh_band{b}_logcount"])
    return np.column_stack(val_feats), np.column_stack(test_feats), names



## 7. Полный 5-fold OOF

Важная особенность: `HashingVectorizer` не обучает словарь и поэтому не передаёт сведения из validation в train. `TfidfTransformer` обучается **отдельно внутри каждого train-fold**. Таким образом IDF также не видит validation/test.

На каждом fold обучаются:

- full URL: SGD hinge;
- full URL: SGD log-loss;
- host/domain-only SGD;
- path/query-only SGD;
- CatBoost на ручных + categorical combination признаках;
- fold-safe reputation;
- fold-safe LSH similarity.

`test.csv` используется только для `transform/predict`. Ни threshold, ни alpha, ни веса stack на test не подбираются.


In [10]:

# 7. 5-fold OOF
X_url = clean["url"].to_numpy()
y = clean["Predicted"].to_numpy(np.int8)
X_test_url = test["url"].to_numpy()

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Статeless hashers — один объект на все folds.
char_hasher = HashingVectorizer(
    analyzer="char", ngram_range=(3,5), n_features=CHAR_HASH_FEATURES,
    alternate_sign=False, norm=None, lowercase=True, dtype=np.float32
)
word_hasher = HashingVectorizer(
    analyzer="word", ngram_range=(1,2), n_features=WORD_HASH_FEATURES,
    alternate_sign=False, norm=None, lowercase=True, dtype=np.float32,
    token_pattern=r"(?u)[A-Za-z0-9_%\-\.]{2,}"
)
host_hasher = HashingVectorizer(
    analyzer="char", ngram_range=(2,5), n_features=HOST_HASH_FEATURES,
    alternate_sign=False, norm=None, lowercase=True, dtype=np.float32
)
path_hasher = HashingVectorizer(
    analyzer="char", ngram_range=(3,5), n_features=PATH_HASH_FEATURES,
    alternate_sign=False, norm=None, lowercase=True, dtype=np.float32
)

base_names = ["text_hinge","text_log","host_model","path_model","catboost"]
oof_scores = {name: np.zeros(len(clean), dtype=np.float32) for name in base_names}
test_scores = {name: np.zeros(len(test), dtype=np.float64) for name in base_names}

rep_oof = None
rep_test_sum = None
rep_names = None
lsh_oof = None
lsh_test_sum = None
lsh_names = None

# fold artifacts сохраняются для воспроизводимого PKL.
fold_artifacts = []
cat_feature_names = ["host","root","tld","first_path","path_template","host_first","root_path_template"]

cat_train = pd.concat(
    [manual.reset_index(drop=True), parts[cat_feature_names].reset_index(drop=True)],
    axis=1
)
cat_test = pd.concat(
    [manual_test.reset_index(drop=True), parts_test[cat_feature_names].reset_index(drop=True)],
    axis=1
)
for c in cat_feature_names:
    cat_train[c] = cat_train[c].astype(str)
    cat_test[c] = cat_test[c].astype(str)

cat_feature_indices = [cat_train.columns.get_loc(c) for c in cat_feature_names]
fold_id = np.full(len(clean), -1, dtype=np.int8)
fold_report = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_url, y), start=1):
    fold_id[va_idx] = fold
    started = time.time()

    # ---------- full URL char + word ----------
    # Char counts -> fold-local TF-IDF
    A_c = char_hasher.transform(X_url[tr_idx])
    B_c = char_hasher.transform(X_url[va_idx])
    T_c = char_hasher.transform(X_test_url)
    char_tfidf = TfidfTransformer(sublinear_tf=True, norm="l2")
    A_c = char_tfidf.fit_transform(A_c)
    B_c = char_tfidf.transform(B_c)
    T_c = char_tfidf.transform(T_c)

    # Word counts -> fold-local TF-IDF
    A_w = word_hasher.transform(X_url[tr_idx])
    B_w = word_hasher.transform(X_url[va_idx])
    T_w = word_hasher.transform(X_test_url)
    word_tfidf = TfidfTransformer(sublinear_tf=True, norm="l2")
    A_w = word_tfidf.fit_transform(A_w)
    B_w = word_tfidf.transform(B_w)
    T_w = word_tfidf.transform(T_w)

    from scipy.sparse import hstack
    A = hstack([A_c, A_w], format="csr")
    B = hstack([B_c, B_w], format="csr")
    T = hstack([T_c, T_w], format="csr")

    text_hinge = SGDClassifier(
        loss="hinge", alpha=3e-6, penalty="l2", max_iter=2200, tol=1e-4,
        average=True, random_state=SEED + fold
    )
    text_log = SGDClassifier(
        loss="log_loss", alpha=3e-6, penalty="l2", max_iter=2200, tol=1e-4,
        average=True, random_state=SEED + 100 + fold
    )
    text_hinge.fit(A, y[tr_idx])
    text_log.fit(A, y[tr_idx])

    oof_scores["text_hinge"][va_idx] = text_hinge.decision_function(B)
    oof_scores["text_log"][va_idx] = text_log.decision_function(B)
    test_scores["text_hinge"] += text_hinge.decision_function(T) / N_SPLITS
    test_scores["text_log"] += text_log.decision_function(T) / N_SPLITS

    del A, B, T, A_c, B_c, T_c, A_w, B_w, T_w
    gc.collect()

    # ---------- host/domain model ----------
    A = host_hasher.transform(parts["host"].iloc[tr_idx])
    B = host_hasher.transform(parts["host"].iloc[va_idx])
    T = host_hasher.transform(parts_test["host"])
    host_tfidf = TfidfTransformer(sublinear_tf=True, norm="l2")
    A = host_tfidf.fit_transform(A)
    B = host_tfidf.transform(B)
    T = host_tfidf.transform(T)
    host_model = SGDClassifier(
        loss="hinge", alpha=3e-6, max_iter=1800, tol=1e-4,
        average=True, random_state=SEED + 200 + fold
    )
    host_model.fit(A, y[tr_idx])
    oof_scores["host_model"][va_idx] = host_model.decision_function(B)
    test_scores["host_model"] += host_model.decision_function(T) / N_SPLITS
    del A, B, T
    gc.collect()

    # ---------- path/query model ----------
    A = path_hasher.transform(parts["path_query"].iloc[tr_idx])
    B = path_hasher.transform(parts["path_query"].iloc[va_idx])
    T = path_hasher.transform(parts_test["path_query"])
    path_tfidf = TfidfTransformer(sublinear_tf=True, norm="l2")
    A = path_tfidf.fit_transform(A)
    B = path_tfidf.transform(B)
    T = path_tfidf.transform(T)
    path_model = SGDClassifier(
        loss="hinge", alpha=3e-6, max_iter=1800, tol=1e-4,
        average=True, random_state=SEED + 300 + fold
    )
    path_model.fit(A, y[tr_idx])
    oof_scores["path_model"][va_idx] = path_model.decision_function(B)
    test_scores["path_model"] += path_model.decision_function(T) / N_SPLITS
    del A, B, T
    gc.collect()

    # ---------- fold-safe reputation ----------
    rep_val, rep_test, rep_names_now = fold_safe_reputation(
        tr_idx, va_idx, y, parts, parts_test
    )
    if rep_oof is None:
        rep_oof = np.zeros((len(clean), rep_val.shape[1]), dtype=np.float32)
        rep_test_sum = np.zeros((len(test), rep_test.shape[1]), dtype=np.float64)
        rep_names = rep_names_now
    rep_oof[va_idx] = rep_val
    rep_test_sum += rep_test / N_SPLITS

    # ---------- fold-safe LSH similarity ----------
    lsh_val, lsh_test, lsh_names_now = fold_safe_lsh(
        tr_idx, va_idx, y, lsh_bands, lsh_bands_test
    )
    if lsh_oof is None:
        lsh_oof = np.zeros((len(clean), lsh_val.shape[1]), dtype=np.float32)
        lsh_test_sum = np.zeros((len(test), lsh_test.shape[1]), dtype=np.float64)
        lsh_names = lsh_names_now
    lsh_oof[va_idx] = lsh_val
    lsh_test_sum += lsh_test / N_SPLITS

    # ---------- deeper CatBoost ----------
    cat_model = CatBoostClassifier(
        iterations=CATBOOST_ITERATIONS,
        depth=CATBOOST_DEPTH,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED + 400 + fold,
        l2_leaf_reg=6.0,
        random_strength=0.35,
        bootstrap_type="Bayesian",
        bagging_temperature=0.5,
        od_type="Iter",
        od_wait=CATBOOST_OD_WAIT,
        verbose=False,
        thread_count=max(1, (os.cpu_count() or 2) - 1),
        allow_writing_files=False,
    )
    cat_model.fit(
        cat_train.iloc[tr_idx], y[tr_idx],
        cat_features=cat_feature_indices,
        eval_set=(cat_train.iloc[va_idx], y[va_idx]),
        use_best_model=True
    )
    oof_scores["catboost"][va_idx] = cat_model.predict_proba(cat_train.iloc[va_idx])[:, 1]
    test_scores["catboost"] += cat_model.predict_proba(cat_test)[:, 1] / N_SPLITS

    fold_artifacts.append({
        "char_tfidf": char_tfidf,
        "word_tfidf": word_tfidf,
        "text_hinge": text_hinge,
        "text_log": text_log,
        "host_tfidf": host_tfidf,
        "host_model": host_model,
        "path_tfidf": path_tfidf,
        "path_model": path_model,
        "catboost": cat_model,
    })

    row = {
        "fold": fold,
        "seconds": round(time.time() - started, 1),
        "cat_best_iteration": int(cat_model.get_best_iteration()),
    }
    for name in base_names:
        score = oof_scores[name][va_idx]
        pred = score >= (0.5 if name == "catboost" else 0.0)
        row[f"f1_{name}"] = f1_score(y[va_idx], pred)
    fold_report.append(row)
    print(row)

    gc.collect()

fold_report = pd.DataFrame(fold_report)
display(fold_report)


{'fold': 1, 'seconds': 439.1, 'cat_best_iteration': 858, 'f1_text_hinge': 0.9496178630301214, 'f1_text_log': 0.9480056339716504, 'f1_host_model': 0.9344465619989715, 'f1_path_model': 0.8991271433010367, 'f1_catboost': 0.9674879198234206}
{'fold': 2, 'seconds': 438.2, 'cat_best_iteration': 1077, 'f1_text_hinge': 0.9520098318395731, 'f1_text_log': 0.9494725485495085, 'f1_host_model': 0.9369380242592481, 'f1_path_model': 0.8959837545126353, 'f1_catboost': 0.9673615247170935}
{'fold': 3, 'seconds': 556.3, 'cat_best_iteration': 1373, 'f1_text_hinge': 0.9497364006709801, 'f1_text_log': 0.9489090419322977, 'f1_host_model': 0.9345737932701021, 'f1_path_model': 0.8980604420387912, 'f1_catboost': 0.9688775054352682}
{'fold': 4, 'seconds': 372.0, 'cat_best_iteration': 844, 'f1_text_hinge': 0.9496981891348089, 'f1_text_log': 0.9484548454845485, 'f1_host_model': 0.9342332253866409, 'f1_path_model': 0.8995749504108813, 'f1_catboost': 0.9664505721028889}
{'fold': 5, 'seconds': 395.0, 'cat_best_iterat

,fold,seconds,cat_best_iteration,f1_text_hinge,f1_text_log,f1_host_model,f1_path_model,f1_catboost
0,1,439.1,858,0.949618,0.948006,0.934447,0.899127,0.967488
1,2,438.2,1077,0.952010,0.949473,0.936938,0.895984,0.967362
2,3,556.3,1373,0.949736,0.948909,0.934574,0.898060,0.968878
3,4,372.0,844,0.949698,0.948455,0.934233,0.899575,0.966451
4,5,395.0,743,0.949184,0.947476,0.933882,0.896745,0.965955


In [11]:

# 8. Базовые OOF-метрики и pooled thresholds
base_metrics = []
base_thresholds = {}

for name in base_names:
    s = oof_scores[name]
    thr, f1 = best_f1_threshold(y, s)
    base_thresholds[name] = thr
    pred = (s >= thr).astype(int)
    base_metrics.append({
        "model": name,
        "pooled_oof_f1": f1,
        "precision": precision_score(y, pred),
        "recall": recall_score(y, pred),
        "accuracy": accuracy_score(y, pred),
        "roc_auc": roc_auc_score(y, s),
        "threshold": thr,
    })

base_metrics = pd.DataFrame(base_metrics).sort_values("pooled_oof_f1", ascending=False)
display(base_metrics)


,model,pooled_oof_f1,precision,recall,accuracy,roc_auc,threshold
4,catboost,0.967810,0.960194,0.975548,0.969018,0.992335,0.383104
0,text_hinge,0.950805,0.946226,0.955429,0.952798,0.984862,-0.132051
1,text_log,0.949734,0.946744,0.952744,0.951852,0.984920,-0.197379
2,host_model,0.940332,0.927799,0.953209,0.942247,0.977778,-0.403336
3,path_model,0.898123,0.855044,0.945774,0.897563,0.941817,0.044862



## 9. OOF meta-stack

Stack получает **только OOF-предсказания** базовых моделей и fold-safe признаки. Затем сама meta-модель снова обучается в 5-fold режиме, поэтому итоговый `meta_oof` не является in-sample прогнозом.

В test передаются усреднённые base scores от 5 fold-моделей. Threshold выбирается **один раз по pooled `meta_oof`**, а не по public leaderboard.


In [12]:

# 9. Матрица признаков для meta stack
base_oof_matrix = np.column_stack([
    oof_scores["text_hinge"],
    oof_scores["text_log"],
    oof_scores["host_model"],
    oof_scores["path_model"],
    oof_scores["catboost"],
])
base_test_matrix = np.column_stack([
    test_scores["text_hinge"],
    test_scores["text_log"],
    test_scores["host_model"],
    test_scores["path_model"],
    test_scores["catboost"],
])

meta_X = np.column_stack([base_oof_matrix, rep_oof, lsh_oof]).astype(np.float32)
meta_T = np.column_stack([base_test_matrix, rep_test_sum, lsh_test_sum]).astype(np.float32)

meta_feature_names = base_names + rep_names + lsh_names
assert meta_X.shape[1] == len(meta_feature_names)

meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 999)
meta_oof = np.zeros(len(clean), dtype=np.float64)
meta_test = np.zeros(len(test), dtype=np.float64)
meta_models = []

for fold, (tr_idx, va_idx) in enumerate(meta_skf.split(meta_X, y), start=1):
    meta_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.7,
            max_iter=2500,
            class_weight=None,
            random_state=SEED + fold
        )
    )
    meta_model.fit(meta_X[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta_model.predict_proba(meta_X[va_idx])[:, 1]
    meta_test += meta_model.predict_proba(meta_T)[:, 1] / 5
    meta_models.append(meta_model)

stack_threshold, stack_f1 = best_f1_threshold(y, meta_oof)
stack_pred_oof = (meta_oof >= stack_threshold).astype(int)

print("STACK pooled OOF F1:", stack_f1)
print("threshold:", stack_threshold)
print("precision:", precision_score(y, stack_pred_oof))
print("recall:", recall_score(y, stack_pred_oof))
print("accuracy:", accuracy_score(y, stack_pred_oof))
print("ROC-AUC:", roc_auc_score(y, meta_oof))
print("confusion matrix:\n", confusion_matrix(y, stack_pred_oof))


STACK pooled OOF F1: 0.9712959004322829
threshold: 0.4338326156139374
precision: 0.9666351495095142
recall: 0.9760018138857729
accuracy: 0.972459292852178
ROC-AUC: 0.991380764533209
confusion matrix:
 [[88901  2823]
 [ 2011 81787]]



## 10. Вторая устойчивая модель для private

Чтобы не выбирать две почти одинаковые submission по public-score, строим альтернативу с другой структурой ошибок: сильный full-URL text score + CatBoost + host/path. Веса фиксированы заранее и **не оптимизируются на test**. Threshold — снова только pooled OOF.

Это не попытка «выжать ещё 0.001» из public; цель — диверсификация риска между двумя private submissions.


In [13]:

# 10. Stable text+CatBoost blend
def z_from_oof(oof_score, test_score):
    mu = float(np.mean(oof_score))
    sd = float(np.std(oof_score) + 1e-9)
    return (oof_score - mu) / sd, (test_score - mu) / sd

zh_oof, zh_test = z_from_oof(oof_scores["text_hinge"], test_scores["text_hinge"])
zl_oof, zl_test = z_from_oof(oof_scores["text_log"], test_scores["text_log"])
zhost_oof, zhost_test = z_from_oof(oof_scores["host_model"], test_scores["host_model"])
zpath_oof, zpath_test = z_from_oof(oof_scores["path_model"], test_scores["path_model"])

# CatBoost probability -> logit, clipping avoids infinities.
cb_oof_logit = logit(np.clip(oof_scores["catboost"], 1e-5, 1 - 1e-5))
cb_test_logit = logit(np.clip(test_scores["catboost"], 1e-5, 1 - 1e-5))
zcb_oof, zcb_test = z_from_oof(cb_oof_logit, cb_test_logit)

stable_oof = (
    0.50 * zh_oof +
    0.10 * zl_oof +
    0.12 * zhost_oof +
    0.08 * zpath_oof +
    0.20 * zcb_oof
)
stable_test = (
    0.50 * zh_test +
    0.10 * zl_test +
    0.12 * zhost_test +
    0.08 * zpath_test +
    0.20 * zcb_test
)

stable_threshold, stable_f1 = best_f1_threshold(y, stable_oof)
stable_pred_oof = (stable_oof >= stable_threshold).astype(int)

print("STABLE pooled OOF F1:", stable_f1)
print("threshold:", stable_threshold)
print("precision:", precision_score(y, stable_pred_oof))
print("recall:", recall_score(y, stable_pred_oof))
print("ROC-AUC:", roc_auc_score(y, stable_oof))


STABLE pooled OOF F1: 0.9653000976120747
threshold: 0.02802497148513794
precision: 0.9629159047189303
recall: 0.9676961263991981
ROC-AUC: 0.9902820943724668


In [14]:

# 11. Stability по исходным 5 folds + различие ошибок двух финальных решений
stability_rows = []
for fold in range(1, N_SPLITS + 1):
    mask = fold_id == fold
    for name, score, threshold in [
        ("V8_stack", meta_oof, stack_threshold),
        ("V8_textcat_stable", stable_oof, stable_threshold),
    ]:
        pred = score[mask] >= threshold
        stability_rows.append({
            "fold": fold,
            "model": name,
            "f1": f1_score(y[mask], pred),
            "precision": precision_score(y[mask], pred),
            "recall": recall_score(y[mask], pred),
        })

stability = pd.DataFrame(stability_rows)
display(stability)
display(
    stability.groupby("model")[["f1","precision","recall"]]
             .agg(["mean","std","min","max"])
)

disagreement = np.mean(
    (meta_oof >= stack_threshold) != (stable_oof >= stable_threshold)
)
print("OOF disagreement между двумя финальными решениями:", round(float(disagreement), 5))


,fold,model,f1,precision,recall
0,1,V8_stack,0.971748,0.967694,0.975835
1,1,V8_textcat_stable,0.965328,0.963007,0.967661
2,2,V8_stack,0.971091,0.965168,0.977088
3,2,V8_textcat_stable,0.965591,0.962585,0.968616
4,3,V8_stack,0.972391,0.966629,0.978222
5,3,V8_textcat_stable,0.965581,0.962035,0.969153
6,4,V8_stack,0.970939,0.968031,0.973865
7,4,V8_textcat_stable,0.965647,0.965186,0.966108
8,5,V8_stack,0.970309,0.965664,0.974999
9,5,V8_textcat_stable,0.964354,0.961778,0.966943


f1                               precision            \
                       mean       std       min       max      mean       std   
model                                                                           
V8_stack           0.971296  0.000798  0.970309  0.972391  0.966637  0.001242   
V8_textcat_stable  0.965300  0.000543  0.964354  0.965647  0.962918  0.001355   

                                         recall                                
                        min       max      mean       std       min       max  
model                                                                          
V8_stack           0.965168  0.968031  0.976002  0.001711  0.973865  0.978222  
V8_textcat_stable  0.961778  0.965186  0.967696  0.001231  0.966108  0.969153

OOF disagreement между двумя финальными решениями: 0.01563


In [15]:

# 12. Final test predictions + deterministic exact lookup
stack_test_pred = (meta_test >= stack_threshold).astype(int)
stable_test_pred = (stable_test >= stable_threshold).astype(int)

overlap_mask = test["url"].isin(exact_lookup)
print("Однозначных exact train/test URL:", int(overlap_mask.sum()))

# Exact lookup — только после ML inference.
for i in np.flatnonzero(overlap_mask.to_numpy()):
    known = exact_lookup[test.iloc[i]["url"]]
    stack_test_pred[i] = known
    stable_test_pred[i] = known

submission_stack = sample_submission[["Id"]].copy()
submission_stack["Predicted"] = stack_test_pred.astype(int)

submission_stable = sample_submission[["Id"]].copy()
submission_stable["Predicted"] = stable_test_pred.astype(int)

assert len(submission_stack) == len(test) == len(sample_submission)
assert submission_stack["Id"].equals(sample_submission["Id"])
assert submission_stable["Id"].equals(sample_submission["Id"])
assert set(submission_stack["Predicted"].unique()).issubset({0,1})
assert set(submission_stable["Predicted"].unique()).issubset({0,1})
assert not submission_stack.isna().any().any()
assert not submission_stable.isna().any().any()

submission_stack.to_csv("submission_v8_stack.csv", index=False)
submission_stable.to_csv("submission_v8_textcat_stable.csv", index=False)

print("Готово:")
print(" submission_v8_stack.csv")
print(" submission_v8_textcat_stable.csv")
display(submission_stack.head())


Однозначных exact train/test URL: 4106
Готово:
 submission_v8_stack.csv
 submission_v8_textcat_stable.csv


,Id,Predicted
0,0,1
1,1,1
2,2,0
3,3,1
4,4,0


In [16]:

# 13. Сохранение PKL, соответствующих двум submissions

common_bundle = {
    "version": "V8",
    "python_version": sys.version.split()[0],
    "pandas_version": pd.__version__,
    "config": {
        "n_splits": N_SPLITS,
        "char_hash_features": CHAR_HASH_FEATURES,
        "word_hash_features": WORD_HASH_FEATURES,
        "host_hash_features": HOST_HASH_FEATURES,
        "path_hash_features": PATH_HASH_FEATURES,
        "seed": SEED,
    },
    "hashers": {
        "char": char_hasher,
        "word": word_hasher,
        "host": host_hasher,
        "path": path_hasher,
        "simhash": simhash_vec,
    },
    "fold_artifacts": fold_artifacts,
    "meta_models": meta_models,
    "meta_feature_names": meta_feature_names,
    "base_thresholds": base_thresholds,
    "exact_lookup": exact_lookup,
    "brand_list": BRANDS,
    "suspicious_words": SUSPICIOUS_WORDS,
    "notes": (
        "5-fold OOF. Threshold/hyperparameters never selected on test.csv. "
        "Exact URL lookup is applied only after model inference."
    ),
}

stack_bundle = dict(common_bundle)
stack_bundle.update({
    "mode": "stack",
    "threshold": float(stack_threshold),
    "oof_f1": float(stack_f1),
})

stable_bundle = dict(common_bundle)
stable_bundle.update({
    "mode": "textcat_stable",
    "threshold": float(stable_threshold),
    "oof_f1": float(stable_f1),
    "blend_weights": {
        "text_hinge": 0.50,
        "text_log": 0.10,
        "host_model": 0.12,
        "path_model": 0.08,
        "catboost": 0.20,
    },
})

with open("best_phishing_model_v8_stack.pkl", "wb") as f:
    pickle.dump(stack_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

with open("phishing_model_v8_textcat_stable.pkl", "wb") as f:
    pickle.dump(stable_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("PKL сохранены.")


PKL сохранены.


In [17]:

# 14. Финальная сводка
final_summary = pd.DataFrame([
    {
        "solution": "V8_stack",
        "OOF_F1": stack_f1,
        "threshold": stack_threshold,
        "mean_fold_F1": stability.query("model == 'V8_stack'")["f1"].mean(),
        "std_fold_F1": stability.query("model == 'V8_stack'")["f1"].std(),
        "submission": "submission_v8_stack.csv",
        "pkl": "best_phishing_model_v8_stack.pkl",
    },
    {
        "solution": "V8_textcat_stable",
        "OOF_F1": stable_f1,
        "threshold": stable_threshold,
        "mean_fold_F1": stability.query("model == 'V8_textcat_stable'")["f1"].mean(),
        "std_fold_F1": stability.query("model == 'V8_textcat_stable'")["f1"].std(),
        "submission": "submission_v8_textcat_stable.csv",
        "pkl": "phishing_model_v8_textcat_stable.pkl",
    },
]).sort_values("OOF_F1", ascending=False)

display(final_summary)

print(
    "\nДля private имеет смысл отметить именно эти два решения, "
    "если оба показывают устойчивые OOF-метрики и заметное disagreement. "
    "Не выбирать два файла только потому, что они оказались первыми на public leaderboard."
)


,solution,OOF_F1,threshold,mean_fold_F1,std_fold_F1,submission,pkl
0,V8_stack,0.971296,0.433833,0.971296,0.000798,submission_v8_stack.csv,best_phishing_model_v8_stack.pkl
1,V8_textcat_stable,0.965300,0.028025,0.965300,0.000543,submission_v8_textcat_stable.csv,phishing_model_v8_textcat_stable.pkl



Для private имеет смысл отметить именно эти два решения, если оба показывают устойчивые OOF-метрики и заметное disagreement. Не выбирать два файла только потому, что они оказались первыми на public leaderboard.



## 15. Что проверять после запуска

1. Смотрим не только на `pooled OOF F1`, но и на `std_fold_F1`.
2. Если V8-stack даёт высокий средний F1, но большой разброс folds, а stable-модель немного слабее, но устойчивее — это нормальная пара для private.
3. Если similarity/reputation признаки резко поднимают random-stratified OOF, полезно дополнительно сделать диагностический **GroupKFold по root domain**. Он будет существенно строже и измерит перенос на совершенно новые домены.
4. Не оптимизируем threshold, веса blend или hyperparameters по `test.csv` и public leaderboard.
5. Если памяти хватает, следующий шаг — увеличить char hashing с `2**18=262144` до `300_000` через обычный `TfidfVectorizer(max_features=300_000)` и сравнить честный pooled OOF. `HashingVectorizer` специально оставлен здесь, чтобы пять fold-моделей и PKL не раздувались из-за пяти словарей.
6. ARIMA/SARIMA не включены в финальный ensemble: в исходных данных нет временной оси, поэтому это не модель классификации URL
